In [1]:
!pip install datasets

In [2]:
!pip install requests

In [3]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [4]:
# Get the first product in your dataframe
sample_product = products_df.iloc[0]

print("--- Product Information ---")
# Print the metadata (ignoring the raw image object so it doesn't flood the console)
for key, value in sample_product.items():
    if key != 'image': 
        print(f"{key}: {value}")

print("\n--- Image Verification ---")
# HuggingFace dataset stores the image directly as a PIL object under the 'image' key
img = sample_product['image']
print(f"Success! Image loaded. Size: {img.size}")
img.show()  # Opens the image in your system's default viewer

--- Product Information ---
id: 15970
gender: Men
masterCategory: Apparel
subCategory: Topwear
articleType: Shirts
baseColour: Navy Blue
season: Fall
year: 2011.0
usage: Casual
productDisplayName: Turtle Check Men Navy Blue Shirt

--- Image Verification ---
Success! Image loaded. Size: (60, 80)


In [5]:
import base64
import io

def encode_pil_to_base64(pil_image):
    """Encode a PIL Image object to a base64 string."""
    # Create an in-memory byte buffer
    buffered = io.BytesIO()
    
    # Convert image to RGB (in case it has an alpha channel) and save it to the buffer as JPEG
    pil_image.convert('RGB').save(buffered, format="JPEG")
    
    # Get the byte data and encode it to base64
    encoded = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return encoded

# Example usage using the HuggingFace 'image' column
sample_image = products_df.iloc[0]["image"]
encoded_image = encode_pil_to_base64(sample_image)

print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [6]:
placeholder_prices = {
       "Apparel": 29.99,
       "Accessories": 24.99,
       "Footwear": 59.99,
       "Personal Care": 14.99,
   }

image_paths = []
prices = []

for index, row in products_df.iterrows():
       image_path = images_dir / f"{row['id']}.jpg"
       row["image"].convert("RGB").save(image_path)
       image_paths.append(str(image_path))
       prices.append(placeholder_prices.get(row["masterCategory"], 19.99))

products_df["image_path"] = image_paths
products_df["price"] = prices
products_df["name"] = products_df["productDisplayName"]
products_df["category"] = products_df["masterCategory"] + " - " + products_df["articleType"]

print(f"✓ Saved {len(image_paths)} images to '{images_dir}'")

✓ Saved 100 images to 'product_images'


In [7]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """Create a prompt for generating product listings."""
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2"
}}

Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt

# Extracting inputs from your Hugging Face dataframe
sample_product = products_df.iloc[0]

# Mapping the specific Hugging Face dataset columns to the required inputs
name_input = sample_product.get("productDisplayName", "Unknown Product")
category_input = sample_product.get("articleType", "Apparel")

# The Hugging Face dataset does not include a price column, so we assign a placeholder
price_input = 49.99 

# Combining other descriptive columns for the additional_info parameter
color = sample_product.get("baseColour", "Unknown color")
season = sample_product.get("season", "All season")
usage = sample_product.get("usage", "Casual")
additional_input = f"Color: {color}, Season: {season}, Occasion: {usage}"

# Generate and test the prompt
test_prompt = create_product_listing_prompt(
    product_name=name_input,
    price=price_input,
    category=category_input,
    additional_info=additional_input
)

print(test_prompt)

You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Turtle Check Men Navy Blue Shirt
- Price: $49.99
- Category: Shirts
- Additional Info: Color: Navy Blue, Season: Fall, Occasion: Casual

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2"
}

Be specific about what you see in the image. Mention colors, materials, design elements, and

In [8]:
import os
from openai import OpenAI

# Read the API key from the environment variable (NOT written in the code)
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    print("⚠ API key not found! Please set the OPENAI_API_KEY environment variable.")
else:
    client = OpenAI(api_key=api_key)
    print("✓ OpenAI client created successfully")

✓ OpenAI client created successfully


In [9]:
import json

def generate_product_listing(prompt, base64_image):
    """Send prompt and image to GPT-4o and return the parsed JSON."""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            response_format={"type": "json_object"},  # forces valid JSON
            max_tokens=800,                            # was 500 – too short
            temperature=0.7
        )

        raw_content = response.choices[0].message.content
        listing_data = json.loads(raw_content)
        return listing_data

    except Exception as e:
        print(f"API Error: {e}")
        return None


# --- Execution ---

# 1. Pick ONE product and use it for BOTH the prompt and the image
product = products_df.iloc[0]

prompt = create_product_listing_prompt(
    product_name=product["name"],
    price=product["price"],
    category=product["category"],
    additional_info=f"Colour: {product['baseColour']}, Gender: {product['gender']}, Usage: {product['usage']}"
)
base64_image = encode_pil_to_base64(product["image"])          # ✅

# 2. Call the API
print(f"Sending request for: {product['name']} ...")
generated_listing = generate_product_listing(prompt, base64_image)

# 3. Show the result
if generated_listing:
    print("\n--- GENERATED PRODUCT LISTING ---")
    print(f"Title: {generated_listing.get('title')}")
    print(f"\nDescription:\n{generated_listing.get('description')}")

    print("\nKey Features:")
    for feature in generated_listing.get('features', []):
        print(f" - {feature}")

    print(f"\nSEO Keywords: {generated_listing.get('keywords')}")
else:
    print("Failed to generate listing.")

Sending request for: Turtle Check Men Navy Blue Shirt ...

--- GENERATED PRODUCT LISTING ---
Title: Stylish Turtle Check Men Navy Blue Shirt - Casual Wear

Description:
Elevate your casual wardrobe with our Turtle Check Men Navy Blue Shirt, designed for comfort and style. Crafted from high-quality fabric, this shirt features a timeless checkered pattern in navy blue, making it a versatile addition to any outfit. The short sleeves provide breathability, perfect for warm days or layering during cooler weather. With a relaxed fit, it ensures maximum comfort whether you're out with friends or enjoying a laid-back day at home. Pair it with jeans or shorts for a casual look that stands out. Make a statement effortlessly with this must-have shirt, available for just $29.99.

Key Features:
 - Stylish navy blue checkered design
 - High-quality, breathable fabric
 - Short sleeves for comfort in warm weather
 - Relaxed fit for all-day wear
 - Versatile for casual outings and gatherings
 - Easy to

In [10]:
import time
import json

results = []
# The lab requires generated listings for at least 3 products
products_to_process = products_df.head(3) 

print(f"Starting batch processing for {len(products_to_process)} products...\n")

for index, product in products_to_process.iterrows():
    print(f"Processing ({index+1}/{len(products_to_process)}): {product['name']}...")
    
    # 1. Create Prompt
    prompt = create_product_listing_prompt(
        product_name=product["name"],
        price=product["price"],
        category=product["category"],
        additional_info=f"Colour: {product['baseColour']}, Gender: {product['gender']}, Usage: {product['usage']}"
    )
    
    # 2. Encode Image
    base64_image = encode_pil_to_base64(product["image"])
    
    # 3. Call API
    generated_listing = generate_product_listing(prompt, base64_image)
    
    if generated_listing:
        # Attach the original product ID so you can map it back in your database
        generated_listing['product_id'] = int(product['id'])
        results.append(generated_listing)
        print("✓ Success\n")
    else:
        print("⚠ Failed to generate listing\n")
        
    # Implement a small delay to avoid "Rate limit exceeded" errors
    time.sleep(2)

# Save the results to a JSON file for your submission
output_filename = "generated_listings.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"🎉 Batch complete! Saved {len(results)} listings to '{output_filename}'")

Starting batch processing for 3 products...

Processing (1/3): Turtle Check Men Navy Blue Shirt...
✓ Success

Processing (2/3): Peter England Men Party Blue Jeans...
✓ Success

Processing (3/3): Titan Women Silver Watch...
✓ Success

🎉 Batch complete! Saved 3 listings to 'generated_listings.json'


Challenges Faced
During development and testing, a few distinct challenges arose:
Authentication Environment Management: An initial 401 invalid_api_key error occurred because Jupyter Notebooks can sometimes fail to dynamically inherit system environment variables. This was resolved by explicitly managing the API key initialization in the notebook environment.